<a href="https://colab.research.google.com/github/sabrina-pacheco/hello/blob/main/Semin%C3%A1rio_II_Camada_2_(c%C3%B3digo_completo).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
# Utilitário - acima de todas as classes. Ela está por fora, como uma função auxiliar do sistema, disponível para qualquer classe usar.

from datetime import date
import hashlib

def hash_senha(s: str) -> str:
    return hashlib.sha256(s.encode()).hexdigest() # transforma str em bytes - retorna para str hexadecimal.

In [26]:
# Classe Tarefa

class Tarefa:
    def __init__(self, titulo: str, descricao: str, vencimento: date):
        self.titulo = titulo
        self.descricao = descricao
        self.vencimento = vencimento
        self.status = "pendente"        # status inicial
        self.responsaveis = []          # lista de usuários responsáveis

    def iniciar(self) -> None:
        self.status = "em_andamento"

    def concluir(self) -> None:
        self.status = "concluida"

    def atribuir(self, usuario: "Usuario") -> None:
        if usuario not in self.responsaveis:
            self.responsaveis.append(usuario)
        if self not in usuario.tarefas:
            usuario.tarefas.append(self)

    def atualizar(
        self,
        titulo: str = None,
        descricao: str = None,
        vencimento: date = None,
        status: str = None
    ) -> None:
        if titulo is not None:
            self.titulo = titulo
        if descricao is not None:
            self.descricao = descricao
        if vencimento is not None:
            self.vencimento = vencimento
        if status is not None:
            self.status = status



In [27]:
# Classe Projeto

class Projeto:
    def __init__(self, nome: str, descricao: str):
        self.nome = nome
        self.descricao = descricao
        self.tarefas = []    # composição: o projeto contém as tarefas

    def adicionar_tarefa(self, tarefa: "Tarefa") -> None:
        if tarefa not in self.tarefas:
            self.tarefas.append(tarefa)

    def remover_tarefa(self, tarefa: "Tarefa") -> None:
        if tarefa in self.tarefas:
            self.tarefas.remove(tarefa)

    def progresso(self) -> float:
        if not self.tarefas:
            return 0.0
        concluidas = sum(1 for t in self.tarefas if t.status == "concluida")
        return (concluidas / len(self.tarefas)) * 100  # percentual

    def listar_tarefas(self, status: str = None) -> list:
        if status is None:
            return list(self.tarefas)
        return [t for t in self.tarefas if t.status == status]

In [28]:
# Classe Usuário

class Usuario:
    def __init__(self, nome: str, email: str, senha: str):
        self.nome = nome
        self.email = email
        self.senha_hash = hash_senha(senha)
        self.tarefas = []  # tarefas atribuídas a este usuário

    def autenticar(self, email: str, senha: str) -> bool:
        return (
            self.email == email and
            self.senha_hash == hash_senha(senha)
        )

    def alterar_senha(self, senha_atual: str, nova_senha: str) -> None:
        if self.senha_hash != hash_senha(senha_atual):
            raise ValueError("Senha atual incorreta.")
        self.senha_hash = hash_senha(nova_senha)

In [29]:
# Classe Sistema

class Sistema:
    def __init__(self):
        self.usuarios = []
        self.projetos = []

    def registrar_usuario(self, nome: str, email: str, senha: str) -> Usuario:
        for u in self.usuarios: # laço que percorre todos usuários cadastrados na lista.
            if u.email == email:  # se o e-mail é porque já existe um usuário com esse e-mail no sistema.
                raise ValueError("E-mail já cadastrado.")  # O sistema não permita duplicatas.
        novo = Usuario(nome, email, senha)
        self.usuarios.append(novo)  # Senão, o e-mail é adicionado na lista de usuários.
        return novo

    def autenticar_usuario(self, email: str, senha: str) -> Usuario:
        for u in self.usuarios:    #percorre todos usuários já cadastrados e o código verifica cada um deles.
            if u.autenticar(email, senha):
                return u     #autenticação bem sucedida, o usuário está correto.
        raise ValueError("Credenciais inválidas.")   # raise serve para lançar um erro(excessão) de forma intencional.

    def criar_projeto(self, nome: str, descricao: str) -> Projeto:
        for p in self.projetos:   # antes de criar um projeto é importante ver se ele já não existe.
            if p.nome == nome:    # código percorre cada projeto p da lista de projetos cadastrados.
                raise ValueError("Projeto já existe.")
        novo = Projeto(nome, descricao)
        self.projetos.append(novo)
        return novo

    def buscar_projeto(self, nome: str) -> Projeto:
        for p in self.projetos:
            if p.nome == nome:
                return p
        raise LookupError("Projeto não encontrado.") # LookupError é uma exceção padrão do Python. Ela é usada quando algo que você está tentando procurar (lookup) não existe.

    def criar_tarefa(
        self,
        projeto: Projeto,  # projeto onde a tarefa será adicionada.
        titulo: str,
        descricao: str,
        vencimento: date
    ) -> Tarefa:
        nova = Tarefa(titulo, descricao, vencimento)
        projeto.adicionar_tarefa(nova)
        return nova  # função e cria e devolve uma nova tarefa.

    def listar_tarefas_do_usuario(self, usuario: Usuario, status: str = None) -> list:
        if status is None:   # se nenhum status for informado, a função retorna todas as tarefas do usuário.
            return list(usuario.tarefas)
        return [t for t in usuario.tarefas if t.status == status]  # se o status for informado, aplica filtro.
        # Se status = "concluida" → retorna só tarefas concluídas
        # Se status = "pendente" → só tarefas pendentes


In [30]:
from datetime import date


def main():  # Evita que o código rode automaticamente se o arquivo for importado por outro.
    # ==============================
    # 1) Criar o sistema (Sistema.__init__)
    # ==============================
    sistema = Sistema()

    # ==============================
    # 2) Registrar usuários (Sistema.registrar_usuario)
    # ==============================
    ana = sistema.registrar_usuario("Ana", "ana@example.com", "1234")
    bruno = sistema.registrar_usuario("Bruno", "bruno@example.com", "abcd")
    print("Usuários registrados:")
    print("-", ana.nome, ana.email)
    print("-", bruno.nome, bruno.email)

    # ==============================
    # 3) Autenticar usuário (Sistema.autenticar_usuario + Usuario.autenticar)
    # ==============================
    usuario_logado = sistema.autenticar_usuario("ana@example.com", "1234")
    print("\nUsuário autenticado:", usuario_logado.nome)  # \n = quebra de linha.

    # ==============================
    # 4) Alterar senha (Usuario.alterar_senha)
    # ==============================
    print("\nAlterando senha da Ana...")
    usuario_logado.alterar_senha("1234", "nova_senha")  # Testando se a autenticação ainda funciona com a nova senha:
    usuario_logado = sistema.autenticar_usuario("ana@example.com", "nova_senha")
    print("Autenticação com nova senha OK para:", usuario_logado.nome)

    # ==============================
    # 5) Criar projeto (Sistema.criar_projeto, Projeto.__init__)
    # ==============================
    projeto = sistema.criar_projeto(
        "Trabalho POO",
        "Projeto da disciplina de Programação Orientada a Objetos"
    )
    print("\nProjeto criado:", projeto.nome, "-", projeto.descricao)

    # ==============================
    # 6) Criar tarefas (Sistema.criar_tarefa, Tarefa.__init__)
    # ==============================
    tarefa1 = sistema.criar_tarefa(
        projeto,
        "Estudar classes",
        "Ler o material teórico de classes e objetos",
        date(2025, 11, 30)
    )

    tarefa2 = sistema.criar_tarefa(
        projeto,
        "Implementar sistema",
        "Implementar as classes do trabalho",
        date(2025, 12, 5)
    )

    print("\nTarefas criadas para o projeto:")
    for t in projeto.tarefas:
        print(f"- {t.titulo} (status: {t.status})")

    # ==============================
    # 7) Atribuir tarefas (Tarefa.atribuir)
    # ==============================
    tarefa1.atribuir(ana)
    tarefa1.atribuir(bruno)
    tarefa2.atribuir(ana)

    print("\nResponsáveis pelas tarefas:")
    for t in projeto.tarefas:
        nomes_resp = [u.nome for u in t.responsaveis]
        print(f"- {t.titulo}: {', '.join(nomes_resp)}")

    # ==============================
    # 8) Iniciar e concluir tarefa (Tarefa.iniciar, Tarefa.concluir)
    # ==============================
    print("\nMudando status das tarefas...")
    tarefa1.iniciar()
    tarefa1.concluir()   # concluída
    tarefa2.iniciar()    # em andamento

    for t in projeto.tarefas:
        print(f"- {t.titulo} -> status: {t.status}")

    # ==============================
    # 9) Atualizar dados da tarefa (Tarefa.atualizar)
    # ==============================
    print("\nAtualizando a descrição e o status da tarefa 2...")
    tarefa2.atualizar(
        descricao="Implementar todas as classes e testar",
        status="em_andamento"
    )
    print(f"Tarefa 2 atualizada: {tarefa2.titulo} - {tarefa2.descricao} ({tarefa2.status})")

    # ==============================
    # 10) Listar tarefas do projeto (Projeto.listar_tarefas)
    # ==============================
    print("\nTodas as tarefas do projeto:")
    for t in projeto.listar_tarefas():
        print(f"- {t.titulo} ({t.status})")

    print("\nSomente tarefas concluídas:")
    for t in projeto.listar_tarefas(status="concluida"):
        print(f"- {t.titulo} ({t.status})")

    # ==============================
    # 11) Progresso do projeto (Projeto.progresso)
    # ==============================
    print(f"\nProgresso do projeto '{projeto.nome}': {projeto.progresso():.2f}%")

    # ==============================
    # 12) Listar tarefas de um usuário (Sistema.listar_tarefas_do_usuario)
    # ==============================
    print(f"\nTarefas atribuídas à usuária {ana.nome}:")
    for t in sistema.listar_tarefas_do_usuario(ana):
        print(f"- {t.titulo} ({t.status})")

    # ==============================
    # 13) Remover tarefa do projeto (Projeto.remover_tarefa)
    # ==============================
    print("\nRemovendo a tarefa 2 do projeto...")
    projeto.remover_tarefa(tarefa2)

    print("Tarefas que restaram no projeto:")
    for t in projeto.listar_tarefas():
        print(f"- {t.titulo}")

    # ==============================
    # 14) Buscar projeto por nome (Sistema.buscar_projeto)
    # ==============================
    print("\nBuscando projeto pelo nome 'Trabalho POO'...")
    p_encontrado = sistema.buscar_projeto("Trabalho POO")
    print("Projeto encontrado:", p_encontrado.nome)

if __name__ == "__main__":
    main()


Usuários registrados:
- Ana ana@example.com
- Bruno bruno@example.com

Usuário autenticado: Ana

Alterando senha da Ana...
Autenticação com nova senha OK para: Ana

Projeto criado: Trabalho POO - Projeto da disciplina de Programação Orientada a Objetos

Tarefas criadas para o projeto:
- Estudar classes (status: pendente)
- Implementar sistema (status: pendente)

Responsáveis pelas tarefas:
- Estudar classes: Ana, Bruno
- Implementar sistema: Ana

Mudando status das tarefas...
- Estudar classes -> status: concluida
- Implementar sistema -> status: em_andamento

Atualizando a descrição e o status da tarefa 2...
Tarefa 2 atualizada: Implementar sistema - Implementar todas as classes e testar (em_andamento)

Todas as tarefas do projeto:
- Estudar classes (concluida)
- Implementar sistema (em_andamento)

Somente tarefas concluídas:
- Estudar classes (concluida)

Progresso do projeto 'Trabalho POO': 50.00%

Tarefas atribuídas à usuária Ana:
- Estudar classes (concluida)
- Implementar sistema